# Performance of University Students and the Factors Affecting It

**How to run:** keep this notebook inside the `student_performance_project` folder (next to `src/` and `data/`).
Run the cells **one by one, top to bottom** (Shift+Enter), or use **Run → Run All Cells**. Order matters: each step uses the one before it.

| Step | Cell | Run it? |
|---|---|---|
| 0 | Install libraries | **Once only**, skip if already installed |
| 1 | Setup | Every session |
| 2 | A1: load and clean real data | Every session |
| 3 | A2: run Part A analysis | After step 2 |
| 4 | A3: view Part A charts | After step 3 |
| 5 | B1: simulate Indian cohort | Every session |
| 6 | B2: run Part B analysis | After step 5 |
| 7 | B3: view Part B charts | After step 6 |
| 8 | Result tables | Optional |

Part A uses **real** data (CSE students). Part B uses **simulated** data (7 Indian branches).

## Step 0: Install libraries (run ONCE, only if something is missing)

In [ ]:
%pip install pandas numpy scipy statsmodels scikit-learn matplotlib

## Step 1: Setup (run every session)

In [ ]:
import sys, os
from pathlib import Path
from IPython.display import Image, display

# find the project folder (this notebook must sit inside it)
ROOT = Path.cwd()
assert (ROOT / "src").exists(), "Open this notebook from inside the student_performance_project folder."
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 160)
from src.config import FIG, TAB
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
print("Project folder:", ROOT)

## Part A: real CSE student survey
### Step 2 (A1): Load and clean the data

In [ ]:
from src.cleaning import load_and_clean
df, log = load_and_clean()
log.to_csv(TAB / "A_cleaning_log.csv", index=False)
print(log.to_string(index=False))
df.head()

### Step 3 (A2): Run the analysis (about 15 seconds)
Correlations, group gaps, regression and a prediction model. Charts are saved to `outputs/figures`.

In [ ]:
from src.analysis_real import run as run_real
a = run_real(df)
print(f"Students: {a['n']}   mean CGPA: {a['cgpa_mean']:.2f}")
print(f"Regression R2  without prior SGPA: {a['ols']['r2_without_prior']:.2f}   with prior SGPA: {a['ols']['r2_with_prior']:.2f}")
print(f"Prediction R2  without prior SGPA: {a['gbm']['r2_without_prior_mean']:.2f}   with prior SGPA: {a['gbm']['r2_with_prior_mean']:.2f}")

### Step 4 (A3): View the Part A charts

In [ ]:
for f in ["A1_correlations", "A2_group_gaps", "A3_regression", "A4_importance", "A5_dose_response"]:
    display(Image(filename=str(FIG / f"{f}.png")))

## Part B: simulated Indian multi-branch cohort
### Step 5 (B1): Generate the simulated students

In [ ]:
from src.simulate import simulate
sim = simulate()
print(sim.shape)
sim.head()

### Step 6 (B2): Run the analysis (about 10 seconds)

In [ ]:
from src.analysis_india import run as run_india
b = run_india(sim)
print(f"Planted effects recovered inside the 95% interval: {b['mixed']['coverage']:.0%}")
print(f"Prediction R2: {b['gbm_r2']['mean']:.2f}")
print(f"Share of CGPA variance between institutions: {b['mixed']['icc_institution']:.2f}")

### Step 7 (B3): View the Part B charts

In [ ]:
for f in ["B1_branch_cgpa", "B2_recovered_vs_true", "B3_study_by_branch", "B4_importance"]:
    display(Image(filename=str(FIG / f"{f}.png")))

## Step 8 (optional): Look at the result tables

In [ ]:
pd.read_csv(TAB / "A_ols_coefficients.csv", index_col=0).round(3)

In [ ]:
pd.read_csv(TAB / "B_recovered_vs_true.csv").round(3)